# BookVision — Núcleo algorítmico

**TFM: Sistema de Reconocimiento Automático de Libros mediante Visión Artificial y Aprendizaje Profundo**
Antonio Álvarez Velasco — Máster en Ciencia de Datos (UCJC) — dirigido por Anas Ahachad

Este notebook es una **versión reducida y autocontenida** del proyecto completo
(`TFM_BookVision/`), pensada para revisar de un vistazo la parte de ciencia de
datos del trabajo: el algoritmo end-to-end y los experimentos que justifican
sus decisiones de diseño. Deja fuera deliberadamente la ingeniería de
aplicación que no aporta a la evaluación científica del TFM (interfaz
Streamlit, base de datos SQLite, CLI, logging, suite de tests). Esa versión
completa y ejecutable sigue disponible en `../TFM_BookVision/` por si el
tribunal quiere verla funcionando como aplicación real.

## Qué hace el sistema

A partir de una fotografía de la portada de un libro:
1. **Preprocesa** la imagen (OpenCV): endereza, elimina ruido, mejora contraste.
2. **Lee el texto** con OCR (EasyOCR / PaddleOCR).
3. **Extrae campos** (título, autor, ISBN) a partir de las líneas detectadas,
   usando el tamaño de letra y la posición de cada línea.
4. Calcula una **huella visual** de la portada con una CNN preentrenada
   (transfer learning), para reconocer si ya se había visto ese libro.
5. Compara el texto leído con metadatos de **Google Books / Open Library** y
   calcula una puntuación de confianza (*fuzzy matching*).
6. Clasifica el resultado en tres niveles: identificación automática,
   posible coincidencia (revisión manual) o no identificado.

Cada sección de este notebook implementa y ejecuta en vivo una de estas
etapas sobre imágenes reales del dataset sintético del proyecto, y la última
sección reproduce los 5 experimentos de evaluación con sus gráficas.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

# Este notebook asume que se ejecuta con la raíz de este repositorio como
# directorio de trabajo (ábrelo con Jupyter desde ahí).
DATA_DIR = Path("data/synthetic_samples")
RESULTS_DIR = Path("results")
assert DATA_DIR.exists(), (
    "No se encuentra data/synthetic_samples: ejecuta Jupyter desde la raíz "
    "de este repositorio, no desde una subcarpeta."
)

sorted(p.name for p in DATA_DIR.glob("*.png"))


## Datos de ejemplo

El dataset sintético completo (75 imágenes: 15 títulos × 5 condiciones de
captura — frontal, rotada, poca luz, con reflejo, en perspectiva) vive en el
proyecto completo (`TFM_BookVision/data/test/synthetic/`) y se genera con
`src/utils/synthetic_dataset.py`. Aquí se incluye una muestra de **3 títulos ×
5 condiciones = 15 imágenes** suficiente para demostrar cada etapa del
algoritmo sin necesitar el repositorio completo. Los tres títulos son
*Cien años de soledad* (Gabriel García Márquez), *Don Quijote de la Mancha*
(Miguel de Cervantes) y *1984* (George Orwell) — libros reales usados solo
como texto de referencia; las portadas son renders sintéticos, no imágenes
reales con derechos de autor.


---
## 1. Preprocesamiento de imagen (OpenCV)

Antes del OCR, la imagen se pasa por una cadena de operaciones clásicas de
visión artificial: redimensionado, corrección de perspectiva y recorte
automático de la portada (detección de contorno + `warpPerspective`),
corrección de inclinación residual (*deskew*, vía transformada de Hough),
reducción de ruido preservando bordes (filtro bilateral) y mejora de
contraste local (CLAHE). El Experimento 1 (sección 8) cuantifica cuánto
ayuda realmente cada motor OCR con este preprocesado.


In [ ]:
def resize(image, max_dimension=1600):
    h, w = image.shape[:2]
    scale = max_dimension / max(h, w)
    if scale >= 1.0:
        return image
    new_size = (int(w * scale), int(h * scale))
    return cv2.resize(image, new_size, interpolation=cv2.INTER_AREA)


def to_grayscale(image):
    if len(image.shape) == 2:
        return image
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


def denoise(gray):
    return cv2.bilateralFilter(gray, d=7, sigmaColor=50, sigmaSpace=50)


def enhance_contrast(gray):
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    return clahe.apply(gray)


def _order_points(pts):
    # Ordena 4 puntos como (superior-izq, superior-der, inferior-der, inferior-izq)
    pts = pts.reshape(4, 2)
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).flatten()
    tl, br = pts[np.argmin(s)], pts[np.argmax(s)]
    tr, bl = pts[np.argmin(diff)], pts[np.argmax(diff)]
    return np.array([tl, tr, br, bl], dtype="float32")


def _find_cover_quad(image, min_area_ratio=0.25):
    # Busca el contorno rectangular mas grande (la portada) sobre un fondo
    # distinguible; si no lo encuentra (p. ej. la portada ocupa ya todo el
    # encuadre), devuelve None y el resto del pipeline sigue sin recortar.
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.dilate(cv2.Canny(blurred, 40, 120), np.ones((5, 5), np.uint8), iterations=1)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    image_area = image.shape[0] * image.shape[1]
    for c in sorted(contours, key=cv2.contourArea, reverse=True)[:5]:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4 and cv2.contourArea(approx) >= min_area_ratio * image_area:
            return approx
    return None


def autocrop_perspective(image):
    """Detecta el contorno de la portada y corrige perspectiva con
    warpPerspective. Si no se encuentra un contorno rectangular fiable
    (p. ej. fondo poco distinguible), devuelve la imagen original sin
    modificar — no fuerza un recorte de baja confianza."""
    quad = _find_cover_quad(image)
    if quad is None:
        return image, False
    tl, tr, br, bl = _order_points(quad)
    max_width = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    max_height = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))
    dst = np.array([[0, 0], [max_width - 1, 0], [max_width - 1, max_height - 1], [0, max_height - 1]], dtype="float32")
    matrix = cv2.getPerspectiveTransform(np.array([tl, tr, br, bl], dtype="float32"), dst)
    warped = cv2.warpPerspective(image, matrix, (max_width, max_height))
    return warped, True


def estimate_skew_angle(gray):
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(
        edges, 1, np.pi / 180, threshold=100,
        minLineLength=gray.shape[1] // 4, maxLineGap=20,
    )
    if lines is None or len(lines) == 0:
        return 0.0
    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        if x2 == x1:
            continue
        angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        if -45 < angle < 45:
            angles.append(angle)
    return float(np.median(angles)) if angles else 0.0


def deskew(image, angle=None):
    gray = to_grayscale(image)
    if angle is None:
        angle = estimate_skew_angle(gray)
    if abs(angle) < 0.5:
        return image
    h, w = image.shape[:2]
    matrix = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(image, matrix, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


def preprocess_for_ocr(image, *, do_autocrop=True, do_deskew=True, do_denoise=True, do_contrast=True):
    result = resize(image)
    if do_autocrop:
        result, _ = autocrop_perspective(result)
    if do_deskew:
        result = deskew(result)
    gray = to_grayscale(result)
    if do_denoise:
        gray = denoise(gray)
    if do_contrast:
        gray = enhance_contrast(gray)
    return gray


In [ ]:
ejemplo_path = DATA_DIR / "01_rotada.png"
imagen_original = cv2.imread(str(ejemplo_path), cv2.IMREAD_COLOR)
imagen_preprocesada = preprocess_for_ocr(imagen_original)

perspectiva_path = DATA_DIR / "01_perspectiva.png"
imagen_perspectiva = cv2.imread(str(perspectiva_path), cv2.IMREAD_COLOR)
imagen_recortada, encontrado = autocrop_perspective(imagen_perspectiva)

fig, axes = plt.subplots(2, 2, figsize=(9, 9))
axes[0, 0].imshow(cv2.cvtColor(imagen_original, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("Original (condición: rotada)")
axes[0, 1].imshow(imagen_preprocesada, cmap="gray")
axes[0, 1].set_title("Preprocesada (autocrop + deskew + denoise + CLAHE)")
axes[1, 0].imshow(cv2.cvtColor(imagen_perspectiva, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("Original (condición: perspectiva)")
axes[1, 1].imshow(cv2.cvtColor(imagen_recortada, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title(f"Corrección de perspectiva (contorno {'detectado' if encontrado else 'NO detectado'})")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()


---
## 2. OCR: lectura del texto de la portada

El sistema soporta dos motores intercambiables, **EasyOCR** y **PaddleOCR**
(ambos instalables solo con `pip`, a diferencia de Tesseract), para poder
compararlos experimentalmente (Experimento 2). Ejecutar cualquiera de los dos
motores implica cargar un modelo de varios cientos de MB, lo cual no es
necesario para revisar el algoritmo: por eso esta celda reutiliza el texto ya
extraído por el pipeline real (`results/ocr_raw_results.csv`, generado por
`experiments/run_ocr_benchmark.py` sobre las 75 imágenes del dataset
sintético). Si tienes `paddleocr` instalado, la celda siguiente (opcional)
ejecuta el motor en vivo sobre una imagen real.


In [ ]:
ocr_raw = pd.read_csv(RESULTS_DIR / "ocr_raw_results.csv")

ejemplo = ocr_raw[
    (ocr_raw.book_index == 1) & (ocr_raw.condition == "rotada")
    & (ocr_raw.engine == "paddleocr") & (ocr_raw.preprocessing == True)
].iloc[0]

print(f"Título real:      {ejemplo.title_truth}")
print(f"Autor real:       {ejemplo.author_truth}")
print(f"Texto OCR crudo:  {ejemplo.raw_text}")
print(f"Confianza media:  {ejemplo.mean_confidence:.2%}")
print(f"Tiempo OCR:       {ejemplo.elapsed_ocr_s:.3f} s")


In [ ]:
# Celda OPCIONAL: ejecución en vivo de PaddleOCR sobre una imagen real.
# Se ejecuta solo si el paquete está instalado; si no, se omite sin error
# (el resto del notebook no depende de esta celda). Se guardan las líneas
# CON su bounding box (posición de cada esquina) porque la sección 3 las
# necesita para demostrar la extracción de campos basada en layout.
PADDLEOCR_DISPONIBLE = False
try:
    from paddleocr import PaddleOCR
    import time

    reader = PaddleOCR(use_angle_cls=True, lang="latin", show_log=False)
    imagen = cv2.imread(str(DATA_DIR / "01_rotada.png"), cv2.IMREAD_COLOR)  # misma imagen que "ejemplo" más abajo

    t0 = time.perf_counter()
    resultado = reader.ocr(imagen, cls=True)
    elapsed = time.perf_counter() - t0

    paddle_bbox_lines = [
        {"text": texto, "confidence": round(conf, 3), "bbox": bbox}
        for bbox, (texto, conf) in resultado[0]
    ]
    PADDLEOCR_DISPONIBLE = True
    print(f"PaddleOCR real ejecutado en {elapsed:.2f} s. Líneas detectadas:")
    for linea in paddle_bbox_lines:
        print(f"  [{linea['confidence']:.2f}] {linea['text']}")
except Exception as exc:
    print(f"PaddleOCR no disponible en este entorno ({exc!r}): se omite la ejecución en vivo "
          f"(la sección 2 ya muestra el resultado real precomputado; la sección 3 usará "
          f"solo la variante sin layout).")


---
## 3. Extracción de campos: ¿qué línea es el título?

El OCR devuelve líneas de texto sueltas, sin decir cuál es el título y cuál
el autor. La heurística del sistema usa la **posición y el tamaño de letra**
de cada línea (vía el *bounding box* que da el OCR): en una portada, el
título casi siempre está en el cuerpo de letra más grande. Esto es más
robusto que asumir "la línea más larga es el título" — el Experimento 3
(sección 6) demuestra cuantitativamente por qué.

Aquí se muestra la variante que opera solo sobre texto plano (usada cuando no
se dispone de las cajas del OCR, p. ej. al reanalizar texto ya guardado).


In [ ]:
import re
import unicodedata

_PUNCT_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)
_SPACE_RE = re.compile(r"\s+")


def normalize_text(text, remove_accents=True):
    if not text:
        return ""
    result = text.lower().strip()
    if remove_accents:
        result = "".join(c for c in unicodedata.normalize("NFKD", result) if not unicodedata.combining(c))
    result = _PUNCT_RE.sub(" ", result)
    return _SPACE_RE.sub(" ", result).strip()


def is_probable_isbn(text):
    candidates = re.findall(r"(?:ISBN[-:\s]*)?((?:97[89][-\s]?)?(?:\d[-\s]?){9}[\dXx])", text)
    for raw in candidates:
        cleaned = re.sub(r"[-\s]", "", raw).upper()
        if len(cleaned) == 13 and cleaned.isdigit():
            return cleaned
        if len(cleaned) == 10 and cleaned[:-1].isdigit() and cleaned[-1] in "0123456789X":
            return cleaned
    return None


def extract_fields(raw_text):
    # Variante sin layout: la línea más larga como candidata a título.
    lines = [l.strip() for l in raw_text.split("\n") if l.strip()]
    isbn = is_probable_isbn(raw_text)
    if not lines:
        return {"isbn": isbn, "title_guess": None, "author_candidates": []}

    title_candidates = sorted(lines, key=len, reverse=True)[:3]
    title_guess = title_candidates[0]
    author_candidates = [l for l in lines if l != title_guess and 1 <= len(l.split()) <= 4]
    return {"isbn": isbn, "title_guess": title_guess, "author_candidates": author_candidates}


campos = extract_fields(ejemplo.raw_text.replace(" | ", "\n"))
print(f"Título extraído (candidato):   {campos['title_guess']}")
print(f"Candidatos a autor:            {campos['author_candidates']}")
print(f"ISBN detectado:                {campos['isbn']}")


Nótese el fallo típico de esta variante: si la línea del autor es más
larga (en caracteres) que la del título, la heurística de "línea más larga"
la elige por error como título. Esto es precisamente lo que motivó usar el
**tamaño de letra real** (bounding box) en vez de la longitud del texto — la
variante que realmente usa el sistema en producción:


In [ ]:
def _bbox_height(bbox):
    ys = [p[1] for p in bbox]
    return max(ys) - min(ys)


def extract_fields_from_lines(lines):
    # `lines`: lista de dicts {"text": ..., "bbox": [(x, y), ...]}, tal como
    # las devuelve el OCR. El título se identifica por estar en el tamaño de
    # letra más grande (dentro del 25% del máximo), no por ser el texto más
    # largo.
    if not lines:
        return {"title_guess": None, "author_candidates": []}

    heights = [_bbox_height(l["bbox"]) for l in lines]
    max_height = max(heights)

    title_lines = [l for l, h in zip(lines, heights) if h >= 0.75 * max_height]
    title_lines.sort(key=lambda l: min(p[1] for p in l["bbox"]))
    title_guess = " ".join(l["text"].strip() for l in title_lines)

    author_candidates = [
        l["text"].strip() for l in lines
        if l not in title_lines and 1 <= len(l["text"].split()) <= 5
    ]
    return {"title_guess": title_guess, "author_candidates": author_candidates}


if PADDLEOCR_DISPONIBLE:
    campos_layout = extract_fields_from_lines(paddle_bbox_lines)
    print(f"Título extraído (con layout, bbox real): {campos_layout['title_guess']}")
    print(f"Candidatos a autor:                      {campos_layout['author_candidates']}")
else:
    print("Requiere la celda de OCR en vivo de la sección 2 (PaddleOCR no disponible aquí). "
          "Código de referencia: src/ocr/field_extraction.py::extract_fields_from_lines "
          "en el proyecto completo.")


---
## 4. Huella visual de la portada (CNN, *transfer learning*)

El texto es la vía principal para **identificar** un libro nuevo, pero no
sirve bien para **reconocer que ya se había fotografiado antes** ese mismo
ejemplar (ángulo, luz o encuadre distintos). Para eso se usa una CNN
preentrenada en ImageNet (**MobileNetV3-Small**, vía `torchvision`) como
extractor de características: se descarta su clasificador final y se usa el
vector de la penúltima capa como "huella visual" de 576 dimensiones,
comparable por similitud coseno. Se eligió MobileNetV3 frente a alternativas
más pesadas (ResNet50, CLIP) por ser suficientemente discriminativa para
color/textura/composición y ejecutable en CPU en tiempos razonables.


In [ ]:
try:
    import torch
    from PIL import Image
    from torchvision import transforms
    from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

    _PREPROCESS = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    _model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    _model.classifier = torch.nn.Identity()
    _model.eval()

    def extract_embedding(bgr_image):
        rgb = bgr_image[:, :, ::-1]
        pil_image = Image.fromarray(rgb)
        tensor = _PREPROCESS(pil_image).unsqueeze(0)
        with torch.no_grad():
            features = _model(tensor).squeeze(0).numpy()
        norm = np.linalg.norm(features)
        return features / norm if norm > 0 else features

    def cosine_similarity(a, b):
        return float(np.clip(np.dot(a, b), -1.0, 1.0))

    EMBEDDINGS_DISPONIBLE = True
except Exception as exc:
    print(f"PyTorch/torchvision no disponibles en este entorno ({exc!r}): se omite la sección 4.")
    EMBEDDINGS_DISPONIBLE = False


In [ ]:
if EMBEDDINGS_DISPONIBLE:
    pares = {
        "mismo libro, mismo encuadre\n(00_frontal vs 00_perspectiva)": ("00_frontal.png", "00_perspectiva.png"),
        "mismo libro, distinta luz\n(00_frontal vs 00_poca_luz)": ("00_frontal.png", "00_poca_luz.png"),
        "libros distintos\n(00_frontal vs 01_frontal)": ("00_frontal.png", "01_frontal.png"),
        "libros distintos\n(01_frontal vs 02_frontal)": ("01_frontal.png", "02_frontal.png"),
    }

    similitudes = {}
    for etiqueta, (img_a, img_b) in pares.items():
        emb_a = extract_embedding(cv2.imread(str(DATA_DIR / img_a)))
        emb_b = extract_embedding(cv2.imread(str(DATA_DIR / img_b)))
        similitudes[etiqueta] = cosine_similarity(emb_a, emb_b)

    fig, ax = plt.subplots(figsize=(7, 4))
    colores = ["#4c72b0", "#4c72b0", "#c44e52", "#c44e52"]
    ax.barh(list(similitudes.keys()), list(similitudes.values()), color=colores)
    ax.axvline(0.92, color="black", linestyle="--", linewidth=1, label="umbral de duplicado (0.92)")
    ax.set_xlabel("Similitud coseno del embedding visual")
    ax.set_xlim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(
        "Nota: dos portadas sintéticas distintas pueden compartir paleta de color "
        "(ver _PALETTES en synthetic_dataset.py) y dar una similitud visual alta "
        "sin ser el mismo libro — el sistema real exige corroboración textual "
        "antes de aceptar un duplicado puramente visual (ver sección 5)."
    )


---
## 5. Matching de texto y corroboración de duplicados

Para comparar el texto leído por OCR contra un título/autor candidato se usa
**RapidFuzz** (variantes de distancia de Levenshtein en C++), en dos modos:
*fuzzy* (robusto a palabras en distinto orden) y *partial* (robusto a texto
recortado). Esta misma función es la que decide si un duplicado visual
detectado en la sección 4 es fiable, exigiendo que no contradiga claramente
el texto leído.


In [ ]:
try:
    from rapidfuzz import fuzz

    def fuzzy_similarity(a, b):
        a_n, b_n = normalize_text(a), normalize_text(b)
        return fuzz.token_sort_ratio(a_n, b_n) / 100.0 if a_n and b_n else 0.0

    def partial_similarity(a, b):
        a_n, b_n = normalize_text(a), normalize_text(b)
        return fuzz.partial_ratio(a_n, b_n) / 100.0 if a_n and b_n else 0.0

except Exception:
    from difflib import SequenceMatcher

    def fuzzy_similarity(a, b):
        a_n, b_n = normalize_text(a), normalize_text(b)
        return SequenceMatcher(None, a_n, b_n).ratio() if a_n and b_n else 0.0

    partial_similarity = fuzzy_similarity
    print("rapidfuzz no disponible: se usa difflib como aproximación (menos preciso).")


def best_similarity(a, b):
    if not a or not b:
        return 0.0
    return max(fuzzy_similarity(a, b), partial_similarity(a, b))


ejemplos_titulo = [
    ("Cien anos de soledad", "Cien años de soledad"),          # error típico de OCR (tilde)
    ("Don Qijote de la Manca", "Don Quijote de la Mancha"),     # letras perdidas
    ("1984", "1984"),
    ("El Principito", "Cien años de soledad"),                 # libros distintos
]
pd.DataFrame(
    [(a, b, round(best_similarity(a, b), 3)) for a, b in ejemplos_titulo],
    columns=["texto OCR", "título real", "similitud"],
)


---
## 6. Score combinado y umbral de decisión

La puntuación final combina similitud de título y de autor con pesos fijados
en el anteproyecto (Fase 6), ajustables y justificados en
`docs/DOCUMENTACION_TECNICA.md`:

$$\text{score} = 0.6 \cdot \text{sim}(t_{ocr}, t_{cand}) + 0.3 \cdot \text{sim}(a_{ocr}, a_{cand}) + 0.1 \cdot \mathbb{1}[\text{ISBN exacto}]$$

y se clasifica en tres niveles: **≥ 0.90** identificación automática,
**0.70–0.90** posible coincidencia (revisión manual), **< 0.70** no
identificado. A continuación se aplica esta fórmula a las 15 imágenes de
ejemplo, usando el título/autor real como si fuera el candidato que
devolvería Google Books (en la práctica, para libros tan conocidos, la API sí
lo devuelve textualmente).


In [ ]:
WEIGHT_TITLE, WEIGHT_AUTHOR, WEIGHT_OTHER = 0.6, 0.3, 0.1
THRESHOLD_AUTO, THRESHOLD_REVIEW = 0.90, 0.70


def clasificar(score):
    if score >= THRESHOLD_AUTO:
        return "identificado_automatico"
    if score >= THRESHOLD_REVIEW:
        return "posible_coincidencia"
    return "no_identificado"


filas = []
subset = ocr_raw[(ocr_raw.engine == "paddleocr") & (ocr_raw.preprocessing == True)]
for _, fila in subset.iterrows():
    campos = extract_fields(fila.raw_text.replace(" | ", "\n"))
    title_sim = best_similarity(campos["title_guess"], fila.title_truth)
    author_sim = max(
        (best_similarity(c, fila.author_truth) for c in campos["author_candidates"]),
        default=0.0,
    )
    # El dataset sintético no lleva ISBN de referencia, así que este término
    # es 0.0 por construcción en esta demo offline (no hay con qué comparar);
    # la sección 7 sí calcula isbn_match con un candidato real de una API.
    isbn_match = 0.0
    score = round(
        WEIGHT_TITLE * title_sim + WEIGHT_AUTHOR * author_sim + WEIGHT_OTHER * isbn_match,
        3,
    )
    filas.append({
        "libro": fila.title_truth, "condicion": fila.condition,
        "titulo_extraido": campos["title_guess"], "sim_titulo": round(title_sim, 3),
        "sim_autor": round(author_sim, 3), "score": score, "estado": clasificar(score),
    })

tabla_resultados = pd.DataFrame(filas)
tabla_resultados


In [ ]:
tabla_resultados["estado"].value_counts().plot(
    kind="bar", color=["#c44e52", "#dd8452", "#4c72b0"], figsize=(6, 4), rot=0,
)
plt.ylabel("nº de imágenes (de 15)")
plt.title("Estado de identificación — extracción sin layout (demo)")
plt.tight_layout()
plt.show()
print(
    "Nota metodológica: esta demo usa extract_fields() (variante sin bounding "
    "boxes) para que la celda sea autocontenida, y el término ISBN del score "
    "es 0.0 porque el dataset sintético no lleva ISBN de referencia (ver "
    "sección 7 para un ejemplo con los tres términos activos). Los "
    "Experimentos 3 y 4 (sección 8) muestran cuantitativamente que la "
    "extracción sin layout es la parte más frágil del sistema, y motivan por "
    "qué la versión de producción usa extract_fields_from_lines() con la "
    "posición/tamaño real de cada línea del OCR."
)


---
## 7. (Opcional) Consulta en vivo a fuentes de metadatos: score completo con ISBN

El sistema busca candidatos en **Google Books** y, si falla o no encuentra
resultados, en **Open Library** como respaldo — ambas sin necesidad de API
key. Esta celda ejecuta ambas consultas reales (si no hay conexión a
internet o hay un `429 Too Many Requests`, frecuente en Google Books sin
clave, cae de forma controlada a la siguiente fuente) y, con el candidato
que devuelva la que funcione, calcula el **score completo con sus tres
términos** —título, autor e ISBN— sobre un ejemplo simulado de texto OCR,
cerrando así la fórmula que la sección 6 no podía ejercitar por completo al
no haber ISBN de referencia en el dataset sintético.


In [ ]:
import requests


def buscar_google_books(query, timeout=8):
    resp = requests.get(
        "https://www.googleapis.com/books/v1/volumes",
        params={"q": query, "maxResults": 1},
        timeout=timeout,
    )
    resp.raise_for_status()
    items = resp.json().get("items")
    if not items:
        return None
    info = items[0]["volumeInfo"]
    isbn = next(
        (i["identifier"] for i in info.get("industryIdentifiers", [])
         if i["type"] in ("ISBN_13", "ISBN_10")),
        None,
    )
    return {"title": info.get("title"), "authors": info.get("authors", []),
            "isbn": isbn, "fuente": "Google Books"}


def buscar_open_library(query, timeout=8):
    resp = requests.get(
        "https://openlibrary.org/search.json",
        params={"q": query, "limit": 1},
        timeout=timeout,
    )
    resp.raise_for_status()
    docs = resp.json().get("docs")
    if not docs:
        return None
    doc = docs[0]
    isbns = doc.get("isbn", [])
    return {"title": doc.get("title"), "authors": doc.get("author_name", []),
            "isbn": isbns[0] if isbns else None, "fuente": "Open Library"}


def buscar_candidato(query):
    for buscador in (buscar_google_books, buscar_open_library):
        try:
            resultado = buscador(query)
            if resultado is not None:
                return resultado
            print(f"  {buscador.__name__}: sin resultados, probando siguiente fuente...")
        except Exception as exc:
            print(f"  {buscador.__name__} falló ({exc!r}); probando siguiente fuente...")
    return None


query = "Cien años de soledad Gabriel Garcia Marquez"
ocr_titulo_simulado = "CIEN AÑOS DE SOLEDAD"   # lo que el OCR habría leído de esta portada
ocr_autor_simulado = "Gabriel García Márquez"
ocr_isbn_simulado = None                        # esta portada de ejemplo no tiene ISBN legible

candidato = buscar_candidato(query)
if candidato is not None:
    fuente = candidato["fuente"]
    print(f"Candidato encontrado vía {fuente}:")
    print(f"  Título:  {candidato['title']}")
    print(f"  Autor:   {', '.join(candidato['authors'])}")
    print(f"  ISBN:    {candidato['isbn']}")

    title_sim = best_similarity(ocr_titulo_simulado, candidato["title"])
    author_sim = max(
        (best_similarity(ocr_autor_simulado, a) for a in candidato["authors"]),
        default=0.0,
    )
    isbn_match = 1.0 if ocr_isbn_simulado and ocr_isbn_simulado == candidato["isbn"] else 0.0
    score = round(WEIGHT_TITLE * title_sim + WEIGHT_AUTHOR * author_sim + WEIGHT_OTHER * isbn_match, 3)

    print(f"\nsim_titulo={title_sim:.3f}  sim_autor={author_sim:.3f}  isbn_match={isbn_match:.0f}")
    print(f"score = {WEIGHT_TITLE}·{title_sim:.3f} + {WEIGHT_AUTHOR}·{author_sim:.3f} + "
          f"{WEIGHT_OTHER}·{isbn_match:.0f} = {score}  -> {clasificar(score)}")
else:
    print("No se pudo consultar ninguna fuente en este momento (sin conexión o ambas "
          "APIs fallaron). Es un fallo esperado y manejado: el sistema real reintenta "
          "con backoff antes de marcar la imagen como no identificada.")


---
## 8. Resultados experimentales (Fase 10 del anteproyecto)

Cinco experimentos sobre el dataset sintético completo (75 imágenes, 300
ejecuciones de OCR), ya calculados por `experiments/` en el proyecto completo
y guardados en `results/*.csv`. Se cargan aquí en vez de recalcularse para que
este notebook no dependa de instalar ambos motores OCR.


### Experimento 1 — Preprocesamiento OpenCV: con vs. sin

In [ ]:
exp1 = pd.read_csv(RESULTS_DIR / "exp1_preprocessing_summary.csv")
display(exp1)

pivot = exp1.pivot(index="engine", columns="preprocessing", values="cer_mean")
pivot.columns = ["sin preprocesado", "con preprocesado"]
pivot.plot(kind="bar", rot=0, figsize=(6, 4), color=["#c44e52", "#4c72b0"])
plt.ylabel("CER medio (Character Error Rate)")
plt.title("Experimento 1 — efecto del preprocesado en el CER")
plt.tight_layout()
plt.show()


**Interpretación.** El preprocesado reduce drásticamente el error de EasyOCR
(CER −83%, de 0.095 a 0.016). En PaddleOCR el efecto es mucho menor porque ya
incorpora su propia normalización de imagen interna. **Conclusión:** el
preprocesado es claramente beneficioso con EasyOCR y prescindible con
PaddleOCR.

### Experimento 2 — Comparación de motores OCR

In [ ]:
exp2 = pd.read_csv(RESULTS_DIR / "exp2_engines_summary.csv")
display(exp2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(exp2.engine, exp2.cer_mean, color=["#4c72b0", "#55a868"])
axes[0].set_title("CER medio")
axes[1].bar(exp2.engine, exp2.tiempo_medio_s, color=["#4c72b0", "#55a868"])
axes[1].set_title("Tiempo medio por imagen (s)")
plt.tight_layout()
plt.show()


**Interpretación.** PaddleOCR obtiene un CER 4 veces menor que EasyOCR y es
~20 veces más rápido (0.29 s frente a 5.90 s por imagen), con mayor confianza
media. Por eso se fijó **PaddleOCR como motor por defecto** del sistema
(`config.py`), documentando aquí la evidencia que sostiene esa decisión.

### Experimento 3 — Matching exacto vs. *fuzzy*

In [ ]:
exp3 = pd.read_csv(RESULTS_DIR / "exp3_matching_summary.csv")
display(exp3)

x = np.arange(len(exp3))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, exp3.exact_accuracy, width, label="exacto", color="#c44e52")
ax.bar(x + width/2, exp3.fuzzy_accuracy, width, label="fuzzy (≥80%)", color="#4c72b0")
ax.set_xticks(x)
ax.set_xticklabels([f"{r.engine}\n(preproc={r.preprocessing})" for r in exp3.itertuples()])
ax.set_ylabel("accuracy")
ax.legend()
plt.tight_layout()
plt.show()


**Interpretación — resultado inesperado y honesto.** El *fuzzy matching* NO
mejora el matching exacto aquí: este experimento usa deliberadamente una
extracción de título sin información de layout ("la línea más larga"), y
cuando esa heurística elige la línea equivocada, el resultado no es "un
título con pequeños errores" sino un texto completamente distinto. Esto
**confirma cuantitativamente** la necesidad de la extracción basada en
posición/tamaño de letra (sección 3) que sí usa el sistema real.

### Experimento 4 — Solo título vs. título + autor

In [ ]:
exp4a = pd.read_csv(RESULTS_DIR / "exp4_title_vs_title_author_summary.csv")
exp4b = pd.read_csv(RESULTS_DIR / "exp4_detection_rate.csv")
display(exp4a)
display(exp4b)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(exp4a))
width = 0.35
axes[0].bar(x - width/2, exp4a.score_title_only_medio, width, label="solo título", color="#4c72b0")
axes[0].bar(x + width/2, exp4a.score_title_author_medio, width, label="título + autor", color="#c44e52")
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{r.engine}\npreproc={r.preprocessing}" for r in exp4a.itertuples()])
axes[0].set_ylabel("score medio")
axes[0].legend()

axes[1].bar(exp4b.estrategia, exp4b.tasa_automatico, color=["#4c72b0", "#c44e52"])
axes[1].set_ylabel("tasa de identificación automática (≥90%)")
plt.tight_layout()
plt.show()


**Interpretación — otro resultado honesto y contrario a la hipótesis
inicial.** Con la extracción de texto plano (sin bounding boxes) usada en
este experimento, añadir el autor al score lo **empeora** en vez de
mejorarlo, porque el candidato a "autor" extraído así es poco fiable. Esto no
invalida combinar título y autor en general: una prueba manual end-to-end con
la extracción real basada en layout identificó correctamente *Cien años de
soledad* con score 0.90. **Lectura correcta:** el valor de combinar señales
depende críticamente de la calidad de la extracción de campos — el verdadero
cuello de botella del sistema, más que el algoritmo de matching en sí.

### Experimento 5 — Impacto de la condición de captura

In [ ]:
exp5 = pd.read_csv(RESULTS_DIR / "exp5_image_quality_summary.csv")
display(exp5)

exp5.plot(x="condition", y="cer_mean", kind="bar", legend=False, color="#4c72b0", rot=20, figsize=(6, 4))
plt.ylabel("CER medio")
plt.title("Experimento 5 — CER por condición de captura (EasyOCR + preprocesado)")
plt.tight_layout()
plt.show()


**Interpretación.** Las diferencias entre condiciones son pequeñas (CER
entre 0% y 2.1%). Esto es una **limitación del dataset sintético, no una
conclusión general de robustez**: las transformaciones sintéticas (rotación,
oscurecimiento, perspectiva) son moderadas y se aplican sobre texto
renderizado de alto contraste, mucho más "limpio" que una fotografía real. El
resultado a destacar es metodológico: la infraestructura de evaluación
funciona correctamente en las 5 condiciones; medir la degradación real
requiere fotografías reales (evaluadas aparte en el proyecto completo con 253
fotos: accuracy global 7.1%, pero 83.3% cuando el sistema declara
identificación automática — ver la sección 9 de este mismo notebook y
`docs/RESULTADOS_FOTOS_REALES.md` en el proyecto completo).

---
## 9. Calibración del umbral de decisión (validación con fotografías reales)

Los umbrales de decisión (0.90 automático, 0.70 revisión) se fijaron en el
anteproyecto antes de disponer de datos reales, sin una búsqueda experimental
de qué valor ofrece el mejor compromiso entre precisión y cobertura. Esta
sección reproduce esa calibración empírica, ejecutada con
`../TFM_BookVision/experiments/calibrate_threshold.py` sobre las 253
fotografías reales evaluadas con el proyecto completo (ver
`docs/RESULTADOS_FOTOS_REALES.md` ahí): de los 27 casos con identificación
verificada manualmente, se recalcula qué precisión, cobertura y recall
resultarían de mover el umbral entre 0.70 y 0.95, manteniendo fijos los
scores ya calculados por el mismo algoritmo que demuestra este notebook.


In [ ]:
calib = pd.read_csv(RESULTS_DIR / "threshold_calibration.csv")
display(calib)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(calib["umbral"], calib["precision"], marker="o", label="Precisión")
ax.plot(calib["umbral"], calib["coverage"], marker="s", label="Cobertura")
ax.plot(calib["umbral"], calib["recall"], marker="^", label="Recall")
ax.axvline(0.90, color="crimson", linestyle="--", linewidth=1, label="Umbral adoptado (0.90)")
ax.set_xlabel("Umbral de decisión (score)")
ax.set_ylabel("Proporción")
ax.legend(fontsize=8)
ax.set_title("Calibración del umbral sobre 253 fotografías reales")
plt.tight_layout()
plt.show()


**Interpretación.** El F1 óptimo (0.80) se alcanza en 0.70, no en 0.90:
bajar el umbral duplica la cobertura automática (4.7% → 10.7%) y sube el
recall al 100% de los casos verificados, pero la precisión cae del 83.3% al
66.7% — uno de cada tres libros propuestos automáticamente sería entonces
incorrecto. Mantener 0.90 no es "subóptimo" sino una elección basada en el
coste asimétrico de los dos tipos de error: un falso positivo automático
(p. ej. confundir dos libros de la misma saga, ver
`docs/RESULTADOS_FOTOS_REALES.md` en el proyecto completo) no tiene
supervisión humana, mientras que un falso negativo simplemente cae a
revisión manual. Nota de tamaño muestral: con solo 12 y 15 casos verificados
respectivamente, los intervalos de confianza (Wilson, 95%) para las
precisiones de 0.90 y 0.70 son anchos — [55.2%, 95.3%] y [30.1%, 75.2%] — por
lo que ninguna de las dos cifras puntuales debe leerse como un valor exacto.


---
## Síntesis general

1. **PaddleOCR** es preferible a EasyOCR en este dataset (mejor CER, ~20×
   más rápido) → fijado como motor por defecto.
2. El **preprocesamiento OpenCV** aporta una mejora clara con EasyOCR,
   marginal con PaddleOCR.
3. Los Experimentos 3 y 4 revelan, de forma consistente, que **la calidad de
   la extracción de campos (título/autor) es el cuello de botella real del
   sistema** — más determinante que la estrictez del matching o que combinar
   señales — cuando esa extracción no usa información de layout. Esto valida
   la decisión de diseño de usar bounding boxes en producción.
4. El dataset sintético, aunque útil para validar el pipeline de extremo a
   extremo, es demasiado "limpio" para estresar la robustez frente a
   condiciones de captura adversas; la evaluación con fotografías reales
   (sección 9 y proyecto completo) confirma que el cuello de botella final
   está en la fase de matching/APIs, no en el OCR.
5. La calibración del umbral (sección 9) muestra que 0.90 no maximiza F1
   (el óptimo está en 0.70), pero se mantiene por el coste asimétrico de un
   falso positivo automático frente a un falso negativo que cae a revisión
   manual — con intervalos de confianza amplios dado el tamaño muestral
   (12 y 15 casos).

## Dónde está el resto

Este notebook cubre la parte algorítmica, incluida su calibración empírica
sobre datos reales (sección 9). El proyecto completo, en
`../TFM_BookVision/`, añade sobre esto: interfaz Streamlit, persistencia en
SQLite, CLI (`main.py`), suite de 76 tests, generación del dataset sintético,
evaluación con 253 fotografías reales, y toda la documentación de la memoria
(`docs/MEMORIA_TFM.md`, `docs/DOCUMENTACION_TECNICA.md`,
`docs/LIMITACIONES.md`, `docs/RESULTADOS_FOTOS_REALES.md`).
